In [ ]:
import pandas as pd
import numpy as np
import datetime
from scipy.spatial.transform import Rotation as R
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

pd.options.mode.chained_assignment = None

In [ ]:
input_path = r"C:\streaming_emulator\Driver behavior\Data\output\raw_data_combined.csv"

df = pd.read_csv(input_path)

df['Time'] = pd.to_datetime(df['Time'], utc=True)
df = df.sort_values(by=['Trip Number', 'Time']).reset_index(drop=True)

df['Delta_t_sec'] = 4.0

df['Distance_m'] = (df['Speed (GPS)(km/h)'] * (5.0 / 18.0)) * df['Delta_t_sec']
df['Cumulative_Distance_m'] = df.groupby('Trip Number')['Distance_m'].cumsum()

In [ ]:
df['Acc_X_cal'] = df['Acceleration Sensor(X axis)(g)']
df['Acc_Y_cal'] = df['Acceleration Sensor(Y axis)(g)']
df['Acc_Z_cal'] = df['Acceleration Sensor(Z axis)(g)']

for trip_id, trip_data in df.groupby('Trip Number'):
    stationary = trip_data[trip_data['Speed (GPS)(km/h)'] == 0]
    
    if not stationary.empty:
        med_x = stationary['Acceleration Sensor(X axis)(g)'].median()
        med_y = stationary['Acceleration Sensor(Y axis)(g)'].median()
        med_z = stationary['Acceleration Sensor(Z axis)(g)'].median()
    else:
        med_x, med_y, med_z = 0.0, 0.0, 1.0 
        
    pitch = np.arctan2(-med_x, np.sqrt(med_y**2 + med_z**2))
    roll = np.arctan2(med_y, np.sqrt(med_x**2 + med_z**2))
    
    rot = R.from_euler('xy', [-roll, -pitch], degrees=False)
    
    acc_vectors = trip_data[['Acceleration Sensor(X axis)(g)', 'Acceleration Sensor(Y axis)(g)', 'Acceleration Sensor(Z axis)(g)']].values
    calibrated_vectors = rot.apply(acc_vectors)
    
    idx = trip_data.index
    df.loc[idx, 'Acc_X_cal'] = calibrated_vectors[:, 0]
    df.loc[idx, 'Acc_Y_cal'] = calibrated_vectors[:, 1]
    df.loc[idx, 'Acc_Z_cal'] = calibrated_vectors[:, 2]

df['Engine_RPM_Diff'] = df.groupby('Trip Number')['Engine RPM(rpm)'].diff().fillna(0)

df['Event_Harsh_Braking'] = (df['Acc_X_cal'] < -0.4) & (df['Relative Throttle Position(%)'] < 5.0)
df['Event_Harsh_Accel'] = (df['Acc_X_cal'] > 0.3) & (df['Relative Throttle Position(%)'] > 60.0) & (df['Engine_RPM_Diff'] > 0)
df['Event_Harsh_Cornering'] = (df['Acc_Y_cal'].abs() > 0.4) & (df['Speed (GPS)(km/h)'] > 30.0)

df['is_idling_raw'] = (df['Speed (GPS)(km/h)'] == 0) & (df['Engine RPM(rpm)'] > 500)
df['idle_state_change'] = df.groupby('Trip Number')['is_idling_raw'].shift() != df['is_idling_raw']
df['idle_group'] = df.groupby('Trip Number')['idle_state_change'].cumsum()

idle_durations = df[df['is_idling_raw']].groupby(['Trip Number', 'idle_group'])['Delta_t_sec'].transform('sum')
df['Event_Excessive_Idling'] = df['is_idling_raw'] & (idle_durations > 180.0)

df['Valid_GPS'] = (df['Horizontal Dilution of Precision'] <= 2.0) & (df['GPS Accuracy(m)'] < 10.0)

df.drop(columns=['is_idling_raw', 'idle_state_change', 'idle_group', 'Engine_RPM_Diff'], inplace=True)

In [ ]:
output_dir = r"C:\streaming_emulator\Driver behavior\Data\results"
os.makedirs(output_dir, exist_ok=True)

current_time = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"results_{current_time}.csv")

df.to_csv(output_path, index=False)
print(f"Processed dataset saved to: {output_path}")

In [ ]:
fig_gg = px.scatter(
    df, 
    x='Acc_Y_cal', 
    y='Acc_X_cal', 
    color='Speed (GPS)(km/h)',
    title="Vehicle Dynamics: Traction Circle (G-G Diagram)",
    labels={'Acc_Y_cal': 'Lateral G (Cornering)', 'Acc_X_cal': 'Longitudinal G (Accel/Brake)'},
    color_continuous_scale=px.colors.sequential.Viridis,
    opacity=0.6
)

fig_gg.add_shape(type="circle", x0=-1, y0=-1, x1=1, y1=1, line_color="red", line_dash="dash")
fig_gg.add_hline(y=0, line_width=1, line_dash="solid", line_color="black")
fig_gg.add_vline(x=0, line_width=1, line_dash="solid", line_color="black")

fig_gg.update_yaxes(range=[-1.5, 1.5], scaleanchor="x", scaleratio=1)
fig_gg.update_xaxes(range=[-1.5, 1.5])
fig_gg.show()

In [ ]:
sample_trip = df['Trip Number'].iloc[0]
df_trip = df[df['Trip Number'] == sample_trip]

fig_telemetry = make_subplots(
    rows=3, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.05,
    subplot_titles=("Vehicle Speed & Engine RPM", "Throttle Position", "Calibrated G-Forces")
)

fig_telemetry.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Speed (GPS)(km/h)'], name='Speed (km/h)'), row=1, col=1)
fig_telemetry.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Engine RPM(rpm)'], name='RPM', yaxis="y2"), row=1, col=1)

fig_telemetry.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Relative Throttle Position(%)'], name='Throttle %', fill='tozeroy'), row=2, col=1)

fig_telemetry.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Acc_X_cal'], name='Longitudinal G'), row=3, col=1)
fig_telemetry.add_trace(go.Scatter(x=df_trip['Time'], y=df_trip['Acc_Y_cal'], name='Lateral G'), row=3, col=1)

fig_telemetry.update_layout(height=800, title_text=f"Telemetry Timeline for Trip {sample_trip}", hovermode="x unified")
fig_telemetry.show()

In [ ]:
df_valid_gps = df[df['Valid_GPS'] == True].copy()

df_valid_gps['Event_Type'] = 'Normal'
df_valid_gps.loc[df_valid_gps['Event_Harsh_Braking'], 'Event_Type'] = 'Harsh Braking'
df_valid_gps.loc[df_valid_gps['Event_Harsh_Accel'], 'Event_Type'] = 'Harsh Accel'
df_valid_gps.loc[df_valid_gps['Event_Harsh_Cornering'], 'Event_Type'] = 'Harsh Cornering'

fig_map = px.scatter_mapbox(
    df_valid_gps, 
    lat="Latitude", 
    lon="Longitude", 
    color="Event_Type",
    color_discrete_map={
        'Normal': 'blue', 
        'Harsh Braking': 'red', 
        'Harsh Accel': 'green', 
        'Harsh Cornering': 'orange'
    },
    zoom=12, 
    title="Spatial Distribution of Driving Events (HDOP Filtered)"
)

fig_map.update_layout(mapbox_style="open-street-map", height=600)
fig_map.show()